In [3]:
import os, json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from src import Questions
import matplotlib.patches as mpatches

In [4]:
# definitions
models = [
    "gpt_4o",
    "llama31_70B_Inst",
    "Qwen25_72B_Inst",
    "llama31_8B_Inst",
    "Qwen25_7B_Inst",
    "llama32_1B_Inst",
    "Qwen25_1dot5B_Inst"
]
task_types = [
    'mt_and_summarization',
    'syntaxes',
    'knowledge_and_reasoning',
    'semantics',
    'qa_extraction'
]
demography_profiles = [
    "employment_sector",
    "race",
    "education",
    "sex",
    "age",
    "language",
    "nationality_group"
]
SCORED_ANNOTATOR_DIR = "annotation_artefacts/final_annotator_objects"

In [5]:
# creating a dataframe of all of the annotator profiles
questions = Questions().get_questions()

In [10]:
def calculate_diversity_score(list_of_values):
    if len(list_of_values) < 2:
        return 0  # No diversity if less than 2 elements
    
    total = 0
    count = 0
    for i in range(len(list_of_values)):
        for j in range(i + 1, len(list_of_values)):
            total += abs(list_of_values[i] - list_of_values[j])
            count += 1
    return total / count

model_diversity_scores = []
highlight_data = {
    "gpt_4o" : {},
    "llama31_70B_Inst" : {}
}

for model in models:
    model_specific_scores = []
    for annfilename in os.listdir(SCORED_ANNOTATOR_DIR):
        annobj_for_df = {}
        annotator_filepath = os.path.join(SCORED_ANNOTATOR_DIR, annfilename)
        with open(annotator_filepath, "r") as ao:
            annotator_object = json.load(ao)
        
        annobj_for_df["id"] = annotator_object["id"]
        annobj_for_df.update(**annotator_object["profile"])
        
        for i, (question, scores_for_question) in enumerate(annotator_object["answers"].items()):
            if not scores_for_question["answer"]:
                continue
            else:
                for qdi, scores in scores_for_question["downstream_scores"][model].items():
                    annobj_for_df[qdi] = list(scores.values())[0]
        model_specific_scores.append(annobj_for_df)
    score_df = pd.DataFrame(model_specific_scores)
    exclude_cols = ["id"] + demography_profiles
    task_cols = [col for col in score_df.columns if col not in exclude_cols]
    score_df[task_cols] = (score_df[task_cols] - score_df[task_cols].min()) / (score_df[task_cols].max() - score_df[task_cols].min())
    
    demography_profile_means = {}
    for dp in demography_profiles:
        if model in highlight_data and dp not in highlight_data[model]:
            highlight_data[model][dp] = {}

        demo_avgs = {}
        demo_grouped = score_df.groupby([dp])
        for demo_name, group in demo_grouped:
            # implement here for exceptions
            # here
            # implement here for exceptions
            task_scores = []
            for tc in task_cols:
                task_scores.append(group[tc].mean())
            task_average = np.nanmean(task_scores)
            demo_avgs[demo_name[0]] = task_average

            if(model in highlight_data):
                highlight_data[model][dp][demo_name[0]] = task_average
        demography_profile_means[dp] = demo_avgs
    
    diversity_scores = {}
    for dpm_k, dpm_v in demography_profile_means.items():
        diversity_scores[dpm_k] = calculate_diversity_score(list(dpm_v.values()))
    
    model_diversity_scores.append({
        "model" : model,
        **diversity_scores
    })
            

In [7]:
model_diversity_df = pd.DataFrame(model_diversity_scores)
model_diversity_df

,model,employment_sector,race,education,sex,age,language,nationality_group
0,gpt_4o,0.032555,0.023181,0.012563,0.017490,0.019172,0.036781,0.025735
1,llama31_70B_Inst,0.028382,0.019987,0.013151,0.001440,0.017707,0.025899,0.025332
2,Qwen25_72B_Inst,0.031143,0.015809,0.014220,0.002118,0.023527,0.022264,0.025516
3,llama31_8B_Inst,0.027906,0.008379,0.025363,0.006987,0.017343,0.010427,0.008256
4,Qwen25_7B_Inst,0.021677,0.019841,0.011355,0.009000,0.017525,0.025837,0.024499
5,llama32_1B_Inst,0.036830,0.030927,0.021726,0.010996,0.020318,0.021244,0.014881
6,Qwen25_1dot5B_Inst,0.012976,0.009609,0.011464,0.009069,0.013720,0.012186,0.011235


In [12]:
highlight_data

{'gpt_4o': {'employment_sector': {'Engineering-STEM': 0.6202478136363946,
   'Health-STEM': 0.5714159668988272,
   'Humanities': 0.6005848959572557},
  'race': {'Asian': 0.6190512964984535,
   'Black': 0.584280235792421,
   'White': 0.5960238904294823},
  'education': {'Graduate': 0.6019651741054766,
   'Highschool': 0.6149591421782404,
   'Postgraduate': 0.5961145585697443},
  'sex': {'female': 0.5927316721607386, 'male': 0.6102216762812995},
  'age': {'18-30': 0.6151121729104669,
   '31-40': 0.5863546560949768,
   '41-50': 0.5993236998618927},
  'language': {'L1': 0.5918195912457793, 'L2': 0.6286006344932771},
  'nationality_group': {'Africa': 0.5829349789306064,
   'Asia': 0.6309493956082055,
   'Europe': 0.5959629485241251,
   'North America': 0.6063308406837237}},
 'llama31_70B_Inst': {'employment_sector': {'Engineering-STEM': 0.6001792840027657,
   'Health-STEM': 0.5576065286507564,
   'Humanities': 0.5796611579203393},
  'race': {'Asian': 0.5959033597541614,
   'Black': 0.565922